# Plant Disease Competition — RunPod Runbook

Run cells **top to bottom**.

| Step | What | Expected score |
|---|---|---|
| 1 | CLIP zero-shot | 60–75% |
| 2 | SigLIP 2 fine-tuned + TTA | 93–96% |
| 3 | SigLIP 2 + DINOv2 ensemble | 95–97% |

**Before running:** upload `plant-disease-train.zip` and `plant-disease-test.zip` to `/workspace/` via JupyterLab drag-and-drop.

---

## Cell 1 — Install dependencies

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "datasets", "huggingface_hub",
    "accelerate", "scikit-learn", "pillow", "timm"
], check=True)
print('Done.')

## Cell 2 — Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected! Make sure your RunPod pod has a GPU attached.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:  {gpu_name}')
print(f'VRAM: {vram_gb:.0f} GB')

# Auto batch sizes
if vram_gb >= 70:    # A100 80GB / H100
    SIGLIP_BATCH, DINOV2_BATCH = 64, 32
elif vram_gb >= 35:  # A100 40GB
    SIGLIP_BATCH, DINOV2_BATCH = 32, 16
elif vram_gb >= 20:  # RTX 3090 / 4090
    SIGLIP_BATCH, DINOV2_BATCH = 16, 8
else:
    SIGLIP_BATCH, DINOV2_BATCH = 8, 4

print(f'SigLIP2 batch: {SIGLIP_BATCH}')
print(f'DINOv2 batch:  {DINOV2_BATCH}')

## Cell 3 — Clone competition pack

In [ ]:
import os
from pathlib import Path

WORKSPACE = Path('/workspace')
PACK_DIR  = WORKSPACE / 'competition_pack'

if not PACK_DIR.exists():
    print('Cloning competition pack...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/AleksCela/competition_pack.git', str(PACK_DIR)],
        cwd=str(WORKSPACE), check=True
    )
else:
    print('Already cloned, pulling latest...')
    subprocess.run(['git', 'pull'], cwd=str(PACK_DIR))

# Handle nested folder if repo has competition_pack/competition_pack
PACK = None
for candidate in [PACK_DIR, PACK_DIR / 'competition_pack']:
    if (candidate / 'advanced' / 'train_vit_large.py').exists():
        PACK = candidate
        break

if PACK is None:
    raise RuntimeError('Could not find train_vit_large.py — check repo structure')

os.chdir(PACK / 'advanced')
print(f'Pack: {PACK}')
print(f'Working dir: {os.getcwd()}')

## Cell 4 — Unzip data

Make sure you uploaded both zip files to `/workspace/` first.

In [ ]:
import zipfile

DATA_DIR   = WORKSPACE / 'data'
DATA_TRAIN = DATA_DIR / 'train'
DATA_TEST  = DATA_DIR / 'test'

TRAIN_ZIP = WORKSPACE / 'plant-disease-train.zip'
TEST_ZIP  = WORKSPACE / 'plant-disease-test.zip'

# Check zips exist
assert TRAIN_ZIP.exists(), f'Not found: {TRAIN_ZIP} — upload it via JupyterLab'
assert TEST_ZIP.exists(),  f'Not found: {TEST_ZIP}  — upload it via JupyterLab'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_TRAIN.exists():
    print('Unzipping train data...')
    with zipfile.ZipFile(TRAIN_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    print('Train unzipped.')
else:
    print(f'Train data already at {DATA_TRAIN}')

if not DATA_TEST.exists():
    print('Unzipping test data...')
    with zipfile.ZipFile(TEST_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    print('Test unzipped.')
else:
    print(f'Test data already at {DATA_TEST}')

# Verify — skip hidden folders (.DS_Store, __MACOSX etc)
train_classes = [
    d for d in os.listdir(DATA_TRAIN)
    if (DATA_TRAIN / d).is_dir() and not d.startswith('.')
]
test_images = list(DATA_TEST.rglob('*.jpg')) + list(DATA_TEST.rglob('*.JPG')) + list(DATA_TEST.rglob('*.png'))

print(f'\nTrain classes: {len(train_classes)}')  # expect 39
print(f'Test images:   {len(test_images)}')       # expect 10976
print('\nData ready.')

## Cell 5 — SUBMISSION 1: CLIP zero-shot

**Run this first. No training needed. Gets you on the leaderboard while SigLIP trains.**

In [ ]:
SUB_CLIP   = WORKSPACE / 'submission_clip.csv'
SUB_SIGLIP = WORKSPACE / 'submission_siglip2.csv'
SUB_ENS    = WORKSPACE / 'submission_ensemble.csv'
FINAL_SUB  = WORKSPACE / 'submission.csv'

subprocess.run([
    'python', 'clip_zero_shot.py',
    '--test-dir',   str(DATA_TEST),
    '--output-csv', str(SUB_CLIP),
], check=True)

subprocess.run([
    'python', str(PACK / 'final_submission_check.py'),
    '--csv',      str(SUB_CLIP),
    '--test-dir', str(DATA_TEST),
])

import shutil
shutil.copy(SUB_CLIP, FINAL_SUB)
print('\n>>> submission.csv ready. Download it and SUBMIT NOW. <<<')

## Cell 6 — MAIN MODEL: Train SigLIP 2

**This is your real weapon.** ~50-60 min on A100 80GB.
Model saves to `/workspace/siglip2-model/` after every epoch.

In [ ]:
SIGLIP_DIR = WORKSPACE / 'siglip2-model'

subprocess.run([
    'python', 'train_vit_large.py',
    '--model',           'google/siglip2-so400m-patch14-384',
    '--data-dir',        str(DATA_TRAIN),
    '--max-images',      '0',
    '--batch-size',      str(SIGLIP_BATCH),
    '--eval-batch-size', str(SIGLIP_BATCH * 2),
    '--epochs',          '5',
    '--lr',              '1e-5',
    '--weighted-loss',
    '--no-phase1',
    '--output-dir',      str(SIGLIP_DIR),
])
print('SigLIP 2 training done.')

In [ ]:
# FALLBACK — only run if Cell 6 fails with model-not-found
subprocess.run([
    'python', 'train_vit_large.py',
    '--model',           'google/vit-large-patch16-224',
    '--data-dir',        str(DATA_TRAIN),
    '--max-images',      '0',
    '--batch-size',      str(SIGLIP_BATCH),
    '--eval-batch-size', str(SIGLIP_BATCH * 2),
    '--epochs',          '5',
    '--lr',              '2e-5',
    '--weighted-loss',
    '--no-phase1',
    '--output-dir',      str(SIGLIP_DIR),
])
print('ViT-large training done.')

## Cell 7 — SUBMISSION 2: SigLIP 2 TTA + OOD protection

`--ood-threshold 0.5`: anything the model is <50% confident about → classified as `other`.
Plant disease images are typically 95%+ confident. Potholes and junk images are not.

In [ ]:
siglip_best = SIGLIP_DIR / 'best'
assert siglip_best.exists(), f'Model not found at {siglip_best} — did Cell 6 finish?'

subprocess.run([
    'python', 'predict_tta.py',
    '--model-dir',     str(siglip_best),
    '--test-dir',      str(DATA_TEST),
    '--output-csv',    str(SUB_SIGLIP),
    '--num-tta',       '6',
    '--ood-threshold', '0.5',
    '--batch-size',    '16',
])

subprocess.run([
    'python', str(PACK / 'final_submission_check.py'),
    '--csv',       str(SUB_SIGLIP),
    '--test-dir',  str(DATA_TEST),
    '--model-dir', str(siglip_best),
])

shutil.copy(SUB_SIGLIP, FINAL_SUB)

# Show predicted label distribution
import csv
from collections import Counter
with open(FINAL_SUB) as f:
    labels = [r['label'] for r in csv.DictReader(f)]
print(f'\nTotal rows: {len(labels)}')
print('Top predicted classes:')
for lbl, cnt in Counter(labels).most_common(10):
    print(f'  {lbl:50s} {cnt}')

print('\n>>> submission.csv ready. Download and SUBMIT NOW. <<<')

## Cell 8 — SECOND MODEL: Train DINOv2

Start immediately after submitting. Different architecture = different errors = better ensemble.
~55-65 min on A100 80GB.

In [ ]:
DINOV2_DIR = WORKSPACE / 'dinov2-model'

subprocess.run([
    'python', 'train_dinov2.py',
    '--model',      'facebook/dinov2-large',
    '--data-dir',   str(DATA_TRAIN),
    '--max-images', '0',
    '--batch-size', str(DINOV2_BATCH),
    '--epochs',     '5',
    '--lr',         '5e-6',
    '--grad-accum', '2',
    '--output-dir', str(DINOV2_DIR),
])
print('DINOv2 training done.')

## Cell 9 — SUBMISSION 3: Ensemble SigLIP 2 + DINOv2

Averages softmax probabilities. SigLIP 2 weighted 0.6, DINOv2 weighted 0.4.

In [ ]:
dinov2_best = DINOV2_DIR / 'best'
assert siglip_best.exists(),  f'SigLIP model missing: {siglip_best}'
assert dinov2_best.exists(),  f'DINOv2 model missing: {dinov2_best}'

subprocess.run([
    'python', 'ensemble_predict.py',
    '--model-dirs',    str(siglip_best), str(dinov2_best),
    '--model-types',   'hf', 'dinov2',
    '--weights',       '0.6', '0.4',
    '--test-dir',      str(DATA_TEST),
    '--output-csv',    str(SUB_ENS),
    '--ood-threshold', '0.5',
    '--batch-size',    '8',
])

subprocess.run([
    'python', str(PACK / 'final_submission_check.py'),
    '--csv',       str(SUB_ENS),
    '--test-dir',  str(DATA_TEST),
    '--model-dir', str(siglip_best),
])

shutil.copy(SUB_ENS, FINAL_SUB)

with open(FINAL_SUB) as f:
    labels = [r['label'] for r in csv.DictReader(f)]
print(f'Total rows: {len(labels)}')
print('Top predicted classes:')
for lbl, cnt in Counter(labels).most_common(10):
    print(f'  {lbl:50s} {cnt}')

print('\n>>> submission.csv ready. Download and SUBMIT. This is your best shot. <<<')

---
## Cell 10 — Final decision

Go to the leaderboard. Your submissions:

| File | Expected score |
|---|---|
| `submission_clip.csv` | 60–75% |
| `submission_siglip2.csv` | 93–96% |
| `submission_ensemble.csv` | 95–97% |

Pick your highest. If DINOv2 wasn't ready — keep Submission 2. Do NOT submit a half-trained ensemble.

---
## Cell 11 — Troubleshooting

In [ ]:
# Disk and GPU memory status
import shutil as _shutil
total, used, free = _shutil.disk_usage('/workspace')
print(f'Disk: {used/1e9:.1f} / {total/1e9:.1f} GB  ({free/1e9:.1f} GB free)')

import torch
if torch.cuda.is_available():
    alloc    = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f'GPU: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved')

print()
print('OOM fix: halve --batch-size and double --grad-accum')
print('Example: --batch-size 16 --grad-accum 2  (same effective batch as 32)')

In [ ]:
# Free GPU memory between runs if needed
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('GPU memory freed.')